# FMSE Laboratorio 04 - Perfilador de modelos

**Curso:** FMSE 2026 · **Módulo 4:** Mecánica de los modelos fundacionales y economía de la inferencia · **Fase:** Investigar · **Nivel:** Profesional

Página del curso y progreso: [agentic-ai.es/academy/fmse/learn/m04-model-mechanics](https://agentic-ai.es/academy/fmse/learn/m04-model-mechanics).

> **Sobre el idioma.** El código, las claves de los datos y las tareas de ejemplo se quedan en inglés; los mensajes de los validadores están en español. La conclusión puedes escribirla en español.

## 0. Misión y competencia objetivo

**Misión.** Diseñar y ejecutar un mini benchmark de 20 tareas. Entregar una recomendación condicionada a los requisitos, no un "mejor modelo" universal.

**Laboratorio guiado.** Un wrapper independiente del proveedor registra la latencia, el número de tokens, las métricas de éxito y las estimaciones de costo.

**Competencia objetivo (resultados del Módulo 4):**
- Explicar la tokenización, las ventanas de contexto, el prefill, el decode, los tokens y el esfuerzo de razonamiento, y de dónde suele venir la latencia.
- Perfilar el compromiso entre calidad, latencia y costo de distintas configuraciones de modelo.
- Reconocer cuándo ayudan los ejemplos o las instrucciones paso a paso, y cuándo restringen de más el razonamiento.
- Redactar un informe de caracterización del modelo en lugar de elegirlo solo por su fama.

**Guía:** alrededor de un 50% guiado y un 50% por tu cuenta. Es un **reto de ingeniería**: la parte guiada te da un camino de referencia que funciona; el reto, no.

## 1. Requisitos y criterios de aceptación

Los validadores públicos comprueban estos requisitos. Te dicen qué requisito falló y por qué, nunca la implementación que se esperaba.

| ID | Requisito |
| --- | --- |
| PRF-01 | El benchmark contiene >=20 muestras |
| PRF-02 | Se comparan al menos 2 variantes de modelo o de configuración |
| PRF-03 | La métrica de calidad está definida |
| PRF-04 | Se registran la latencia y el costo |
| PRF-05 | La conclusión cita evidencia medida y sus limitaciones |
| PRF-06 | El benchmark es reproducible |

**Control de competencia (portal):** benchmark reproducible; el informe PASA. También necesitas el cuestionario >= 80%, el laboratorio guiado, el artefacto y tu reflexión. Un requisito crítico no se compensa con otras puntuaciones.

## 2. Preparación del entorno

In [ ]:
# Idempotente: se puede volver a ejecutar y nunca imprime secretos.
LAB_ID = "lab-04"
LABKIT_VERSION = "1.1.0"
LABKIT_SOURCES = [
    "https://agentic-ai.es/academy/fmse/labs",  # agentic-ai.es mirror
    "https://raw.githubusercontent.com/farithjhg/fmse-course-labs/main",  # GitHub (public repositories only)
]
import hashlib, json, os, pathlib, sys, urllib.request


def _fmse_bootstrap():
    """Make fmse_labkit importable: installed copy, local checkout, cached download, or verified download."""
    try:
        import fmse_labkit
        if fmse_labkit.__version__ == LABKIT_VERSION:
            return "fmse_labkit ya estaba cargado"
    except ImportError:
        pass
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "fmse_labkit" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            return f"usando el labkit de {folder}"
    target = pathlib.Path("fmse_labkit_dist").resolve()
    marker = target / f".labkit-{LABKIT_VERSION}"
    if marker.exists():
        sys.path.insert(0, str(target))
        return "usando el labkit descargado anteriormente"
    problems = []
    for base in [os.environ.get("FMSE_LABKIT_URL")] + LABKIT_SOURCES:
        if not base:
            continue
        base = base.rstrip("/")
        try:
            manifest = json.loads(urllib.request.urlopen(base + "/manifest.json", timeout=20).read())
            if manifest.get("labkit_version") != LABKIT_VERSION:
                raise RuntimeError(f"mirror serves labkit {manifest.get('labkit_version')}")
            for entry in manifest["files"]:
                if not entry["path"].startswith("fmse_labkit/"):
                    continue
                data = urllib.request.urlopen(f"{base}/{entry['path']}", timeout=20).read()
                if hashlib.sha256(data).hexdigest() != entry["sha256"]:
                    raise RuntimeError(f"integrity check failed for {entry['path']}")
                dest = target / entry["path"]
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(data)
            marker.write_text("ok")
            sys.path.insert(0, str(target))
            return f"labkit descargado y verificado desde {base}"
        except Exception as exc:  # try the next source
            problems.append(f"{base} ({type(exc).__name__})")
    raise RuntimeError(
        "No se pudo cargar fmse_labkit desde: " + "; ".join(problems) + ". "
        "Solución: descarga la carpeta de laboratorios desde agentic-ai.es/academy/fmse/resources, sube la carpeta fmse_labkit junto a este notebook (panel Archivos) y vuelve a ejecutar esta celda."
    )


print(_fmse_bootstrap())
import fmse_labkit as fmse

fmse.set_language("es")
print(f"fmse_labkit {fmse.__version__} listo para {LAB_ID} ({fmse.title(fmse.lab_spec(LAB_ID))})")

## 3. Comprobación de secretos

Este laboratorio funciona completamente sin conexión: no necesitas ninguna clave de proveedor. Las claves que uses en otros laboratorios van en los Secretos de Colab y nunca se imprimen.

In [ ]:
fmse.secrets_check()

## 4. Experimento guiado

`lab04.run_variant` es un wrapper de perfilado independiente del proveedor. Para cada tarea registra si la respuesta es correcta, los tokens de entrada, de salida y de razonamiento, el tiempo hasta el primer token (TTFT), la latencia total y el costo. Los modelos simulados se comportan de forma determinista y documentada; los costos unitarios son cifras hipotéticas que eliges tú.

In [ ]:
from fmse_labkit.labs import lab04
import statistics

guided = fmse.GuidedLog(LAB_ID, required_steps=["profile_short_vs_long", "compare_variants"])
tasks = lab04.task_pool()
unit_costs = {"sim-fast": 0.0004, "sim-reasoning": 0.004, "sim-small": 0.0001}  # hipotéticos, por cada mil tokens
print(len(tasks), "tareas;", sum(t["long_context"] for t in tasks), "son de contexto largo")
print("Escenario:", lab04.SCENARIO)

## 5. Predice antes de ejecutar

Para una misma respuesta corta, ¿cómo cambiará el TTFT entre un prompt corto y uno de contexto largo? ¿Qué variante será la más lenta, y por qué? Escribe tu predicción y después ejecuta el experimento.

In [ ]:
my_prediction = ""
guided.predict("prediction", my_prediction)

In [ ]:
fast = {"name": "fast", "model": "sim-fast", "effort": "low", "prompt_style": "objective"}
for t in (tasks[1], tasks[0]):
    r = lab04.run_variant(fast, t, unit_costs)
    print(f"{t['id']} long_context={t['long_context']}: TTFT {r['ttft_ms']} ms, total {r['latency_ms']} ms, entrada {r['input_tokens']} tokens")
guided.step("profile_short_vs_long")
ids = [t["id"] for t in tasks[:12]]
for v in (fast, {"name": "reasoning", "model": "sim-reasoning", "effort": "high", "prompt_style": "objective"}):
    print(v["name"], lab04.summarize(lab04.profile(v, ids, unit_costs)))
guided.step("compare_variants")
guided.outcome("prediction", "ver la salida de arriba")

**Hoja de trabajo.** Al menos una frase completa por campo: estas observaciones forman parte de la evidencia de tu laboratorio guiado.

In [ ]:
worksheet = {
    "prefill_observation": "",
    "tradeoff_observation": "",
    "prediction_check": "",
}
guided.observe(worksheet)

## 6. Reto

**Mini benchmark e informe de caracterización.** Diseña y ejecuta un mini benchmark de 20 tareas. Entrega una recomendación condicionada a los requisitos, no un "mejor modelo" universal.

Restricciones:
- Los costos unitarios son valores hipotéticos que eliges tú; nunca dejes precios reales fijos en el código.
- La recomendación tiene que citar los ids de los requisitos del escenario.

In [ ]:
my_benchmark = {
    "benchmark": {"task_ids": [], "quality_metric": {"name": "", "definition": ""}},
    "variants": [],  # {"name", "model", "effort", "prompt_style"}
    "unit_costs": unit_costs,
    "results": {},  # nombre de la variante -> lab04.summarize(lab04.profile(variant, task_ids, unit_costs))
    "conclusion": {"recommendation": "", "evidence": "", "limitations": ""},
}

## 7. Validación pública

In [ ]:
submission = my_benchmark
result = fmse.check_public(LAB_ID, submission)
result.show()

## 8. Robustez / pruebas adversariales

Condiciones adversariales que la batería calificada no cubre. No cambian tu puntuación: te enseñan dónde es frágil tu solución.

In [ ]:
probe_results = fmse.probe(LAB_ID, submission)

## 9. Reflexión de ingeniería

Se califica en el portal y se guarda con tu artefacto. Si te sirve, haz aquí un borrador:
- **Decisión:** ¿qué variante recomiendas para cada requisito y qué evidencia medida lo respalda?
- **Compromisos:** ¿a qué renuncia la variante que recomiendas (calidad, latencia o costo) y por qué es aceptable en este caso?
- **Riesgos pendientes:** ¿qué podría hacer que este benchmark resulte engañoso con tráfico real?

In [ ]:
reflection_draft = {"decision": "", "tradeoffs": "", "remaining_risks": ""}

## 10. Exportar el artefacto / resultado de finalización

Exporta el artefacto **Informe de caracterización del modelo fundacional** e imprime un registro de finalización. Pega el registro en la página del Módulo 4 de agentic-ai.es (**Importar el resultado del laboratorio**). El registro es un documento del flujo de aprendizaje, no un certificado firmado.

In [ ]:
artifact = fmse.export_artifact(LAB_ID, my_benchmark, title="Informe de caracterización del modelo fundacional", fmt="json")
record = fmse.make_completion_record(
    LAB_ID,
    result,
    {"reflection_draft": reflection_draft, **{}},
    artifact=artifact,
    guided=guided,
    probes=probe_results,
)
fmse.show_record(record)